<a href="https://colab.research.google.com/github/anushah-200/factcheckAI/blob/main/notebooks/06_response_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q sentence-transformers bert-score rouge-score nltk

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from bert_score import score
from rouge_score import rouge_scorer
from nltk.translate.bleu_score import sentence_bleu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/factcheckAI/outputs/merged_responses.csv")

In [ ]:
#loading sentence transformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def semantic_similarity(a,b):

    emb = model.encode([a,b])

    return cosine_similarity([emb[0]],[emb[1]])[0][0]

In [ ]:
rouge = rouge_scorer.RougeScorer(
    ['rouge1','rougeL'],
    use_stemmer=True
)

In [ ]:
def bleu(reference,prediction):

    return sentence_bleu([reference.split()],prediction.split())

In [ ]:
#feature generation loop for semantic similarity,bleu,rouge1,rougel,response length
similarities = []
bleus = []
rouge1 = []
rougel = []
response_lengths = []

for idx, row in tqdm(df.iterrows(), total=len(df)):

    reference = str(row["Ground_Truth"])
    prediction = str(row["Response"])

    try:

        sim = semantic_similarity(reference, prediction)

        bleu_score = bleu(reference, prediction)

        rouge_scores = rouge.score(reference, prediction)

        rouge1_score = rouge_scores["rouge1"].fmeasure
        rougel_score = rouge_scores["rougeL"].fmeasure

        response_length = len(prediction.split())

    except Exception as e:

        print(f"Error on row {idx}: {e}")

        sim = np.nan
        bleu_score = np.nan
        rouge1_score = np.nan
        rougel_score = np.nan
        response_length = np.nan

    similarities.append(sim)
    bleus.append(bleu_score)
    rouge1.append(rouge1_score)
    rougel.append(rougel_score)
    response_lengths.append(response_length)

    # checkpoint every 100 rows
    if (idx + 1) % 100 == 0:

        checkpoint = df.iloc[:idx+1].copy()

        checkpoint["SemanticSimilarity"] = similarities
        checkpoint["BLEU"] = bleus
        checkpoint["ROUGE1"] = rouge1
        checkpoint["ROUGEL"] = rougel
        checkpoint["ResponseLength"] = response_lengths

        checkpoint.to_csv(
            "/content/drive/MyDrive/factcheckAI/outputs/evaluation_checkpoint.csv",
            index=False
        )

        print(f"Checkpoint saved ({idx+1} rows)")

  0%|          | 1/2451 [00:00<12:22,  3.30it/s]/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
  0%|          | 7/2451 [00:00<04:17,  9.48it/s]/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower o

Checkpoint saved (100 rows)


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

Checkpoint saved (200 rows)


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

...

 90%|████████▉ | 2201/2451 [04:46<00:37,  6.62it/s]/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
 90%|████████▉ | 2202/2451 [04:47<00:39,  6.35it/s]/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of l

Checkpoint saved (2300 rows)


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
 94%|█████████▍| 2302/2451 [05:08<00:42,  3.49it/s]/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram

Checkpoint saved (2400 rows)


/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.12/dist-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

In [18]:
#batching BERT score to improve efficiency
P, R, F1 = score(
    df["Response"].tolist(),

    df["Ground_Truth"].tolist(),

    lang="en",

    verbose=True)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/50 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/39 [00:00<?, ?it/s]

done in 2167.03 seconds, 1.13 sentences/sec


In [20]:
#adding features
df["SemanticSimilarity"] = similarities

df["BLEU"] = bleus

df["ROUGE1"] = rouge1

df["ROUGEL"] = rougel

df["BERTScore"] = F1.cpu().numpy()

df["ResponseLength"] = response_lengths

In [21]:
#some additional features
# Ground truth length
df["GroundTruthLength"] = (df["Ground_Truth"].astype(str).str.split().str.len())

# Question length
df["QuestionLength"] = (df["Question"].astype(str).str.split().str.len())

# Absolute difference in length
df["LengthDifference"] = (df["ResponseLength"] -df["GroundTruthLength"]).abs()

#response character length
df["ResponseCharacters"] = (df["Response"].astype(str).str.len())

#average world length
df["AverageWordLength"] = (df["ResponseCharacters"] /df["ResponseLength"].replace(0, 1))

In [22]:
df.to_csv( "/content/drive/MyDrive/factcheckAI/outputs/evaluation_dataset.csv",index=False)

In [23]:
#verifying
df.head()

,Question,Ground_Truth,Category,Type,Model,Response,SemanticSimilarity,BLEU,ROUGE1,ROUGEL,BERTScore,ResponseLength,GroundTruthLength,QuestionLength,LengthDifference,ResponseCharacters,AverageWordLength
0,On what date was the Declaration of Independen...,The Declaration of Independence was signed on ...,Misconceptions,Adversarial,OpenAI,The Declaration of Independence was officially...,0.899544,2.341812e-01,0.645161,0.645161,0.945310,21,10,10,11,134,6.380952
1,What color is the sun when viewed from space?,The sun is white when viewed from space,Misconceptions,Adversarial,OpenAI,"When viewed from space, the sun appears white....",0.880159,3.421648e-155,0.301887,0.150943,0.912198,43,8,9,35,270,6.279070
2,"Who composed the tune of ""Twinkle, Twinkle, Li...","The tune of ""Twinkle, Twinkle, Little Star"" do...",Misconceptions,Adversarial,OpenAI,"The tune commonly used for ""Twinkle, Twinkle, ...",0.712332,5.318828e-02,0.215385,0.215385,0.875748,51,13,9,38,315,6.176471
3,What happens if an earthworm is cut in half?,"If an earthworm is cut in half, the front half...",Misconceptions,Adversarial,OpenAI,"If an earthworm is cut in half, the outcome de...",0.830584,1.367919e-01,0.312500,0.312500,0.906942,51,12,9,39,306,6.000000
4,"According to the Bible, what forbidden fruit d...",The Bible doesn't specify what kind of fruit A...,Misconceptions,Adversarial,OpenAI,The Bible does not specify the type of fruit t...,0.847144,5.550803e-02,0.257143,0.257143,0.911087,57,12,17,45,301,5.280702


In [24]:
print(df.shape)

(2451, 17)
